In [30]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [31]:
pip install alpaca-trade-api

In [32]:
pip install polygon-api-client

In [33]:
import alpaca_trade_api as tradeapi
from polygon import RESTClient
import pandas as pd
import time
from datetime import datetime, timedelta, timezone
import tensorflow as tf
from tensorflow.keras.models import load_model
import joblib
import numpy as np

In [ ]:
ALPACA_API_KEY = "[...]"
ALPACA_SECRET_KEY = "[...]"
ALPACA_BASE_URL = "https://api.alpaca.markets"

POLYGON_API_KEY = "[...]"

In [35]:
# Inicializa a API Alpaca com as credenciais lidas do arquivo
api = tradeapi.REST(ALPACA_API_KEY, ALPACA_SECRET_KEY, ALPACA_BASE_URL, api_version='v2')

In [36]:
clock = api.get_clock()

In [37]:
clock

Clock({   'is_open': True,
    'next_close': '2025-04-29T16:00:00-04:00',
    'next_open': '2025-04-30T09:30:00-04:00',
    'timestamp': '2025-04-29T13:13:14.204470358-04:00'})

In [38]:
client = RESTClient(POLYGON_API_KEY)

In [39]:
status = client.get_market_status()

In [40]:
status

MarketStatus(after_hours=False, currencies=MarketCurrencies(crypto='open', fx='open'), early_hours=False, exchanges=MarketExchanges(nasdaq='open', nyse='open', otc='open'), indicesGroups=MarketIndices(s_and_p='open', societe_generale='open', cgi='open', msci='open', ftse_russell='open', mstar='open', mstarc='open', cccy='open', nasdaq='open', dow_jones='open'), market='open', server_time='2025-04-29T13:13:14-04:00')

In [41]:
if status.market == "open":
    print("The market is open regular hours.")
elif status.pearly_hours:
    print("The market is open pre hours.")
elif status.after_hours:
    print("The market is open after hours.")
else:
    print("The market is closed.")

The market is open regular hours.


In [42]:
delayed_safe_df = 5

now   = datetime.now(timezone.utc)
start = (now - timedelta(days=delayed_safe_df)).isoformat()

print("Now (UTC):", now)
print("Start (UTC):", start)

Now (UTC): 2025-04-29 17:13:14.609314+00:00
Start (UTC): 2025-04-24T17:13:14.609314+00:00


In [43]:
symbol = "AAPL"
timeframe = "5Min"
data_source = "sip"
dataset_size = 36

In [44]:
# Fetch the historical data
bars = api.get_bars(
    symbol,
    timeframe,
    start,
    feed=data_source
).df.tail(dataset_size)

In [45]:
bars

,close,high,low,trade_count,open,volume,vwap
timestamp,,,,,,,
2025-04-29 14:00:00+00:00,210.8900,211.0000,210.3300,8821,210.8100,413425,210.664221
2025-04-29 14:05:00+00:00,211.5100,211.8400,210.8400,9289,210.8800,546991,211.582703
2025-04-29 14:10:00+00:00,212.0200,212.1090,211.5100,10518,211.5100,758298,211.850167
2025-04-29 14:15:00+00:00,211.2600,212.2400,211.1500,10047,212.0150,605568,211.681098
2025-04-29 14:20:00+00:00,211.0812,211.4100,210.9800,6455,211.2400,376585,211.155843
2025-04-29 14:25:00+00:00,210.9500,211.2200,210.7000,5446,211.0730,317716,210.904705
2025-04-29 14:30:00+00:00,211.1000,211.2000,210.8400,5462,210.9600,284823,211.014102
2025-04-29 14:35:00+00:00,211.3200,211.4900,211.0200,5160,211.1000,249223,211.330045
2025-04-29 14:40:00+00:00,211.2300,211.4498,211.1300,5594,211.3450,289036,211.309857


In [46]:
data = bars[['vwap', 'trade_count']]

In [47]:
data

,vwap,trade_count
timestamp,,
2025-04-29 14:00:00+00:00,210.664221,8821
2025-04-29 14:05:00+00:00,211.582703,9289
2025-04-29 14:10:00+00:00,211.850167,10518
2025-04-29 14:15:00+00:00,211.681098,10047
2025-04-29 14:20:00+00:00,211.155843,6455
2025-04-29 14:25:00+00:00,210.904705,5446
2025-04-29 14:30:00+00:00,211.014102,5462
2025-04-29 14:35:00+00:00,211.330045,5160
2025-04-29 14:40:00+00:00,211.309857,5594


In [ ]:
scaler = joblib.load("[...]")

In [49]:
X_VWAP = data[['vwap']].to_numpy()

X_VWAP_scaled = scaler.transform(X_VWAP)

In [50]:
X_VWAP_scaled

array([[0.28496234],
       [0.28712056],
       [0.28774903],
       [0.28735176],
       [0.28611754],
       [0.28552742],
       [0.28578448],
       [0.28652687],
       [0.28647943],
       [0.28696188],
       [0.2871512 ],
       [0.28780899],
       [0.28714562],
       [0.28677571],
       [0.28603962],
       [0.28571233],
       [0.28535833],
       [0.28573071],
       [0.28495595],
       [0.28516777],
       [0.28574136],
       [0.28579982],
       [0.28532573],
       [0.28500608],
       [0.28534536],
       [0.28613776],
       [0.28590772],
       [0.28516672],
       [0.28492874],
       [0.28462751],
       [0.28440189],
       [0.28436501],
       [0.28449844],
       [0.28362389],
       [0.28330373],
       [0.28247665]])

In [51]:
X_Trade_Count = data[['trade_count']].to_numpy()

In [52]:
X_Trade_Count

array([[ 8821],
       [ 9289],
       [10518],
       [10047],
       [ 6455],
       [ 5446],
       [ 5462],
       [ 5160],
       [ 5594],
       [ 4900],
       [ 4992],
       [ 5105],
       [ 4611],
       [ 3792],
       [ 3965],
       [ 4012],
       [ 4019],
       [ 4400],
       [ 5500],
       [ 3791],
       [ 3241],
       [ 3032],
       [ 2966],
       [ 2988],
       [ 3623],
       [ 3452],
       [ 3707],
       [ 3559],
       [ 2882],
       [ 3723],
       [ 4450],
       [ 3988],
       [ 2973],
       [ 4980],
       [ 3125],
       [ 4893]])

In [53]:
X_combined = np.concatenate([X_VWAP_scaled, X_Trade_Count], axis=1)

In [54]:
X_combined

array([[2.84962344e-01, 8.82100000e+03],
       [2.87120556e-01, 9.28900000e+03],
       [2.87749032e-01, 1.05180000e+04],
       [2.87351760e-01, 1.00470000e+04],
       [2.86117538e-01, 6.45500000e+03],
       [2.85527424e-01, 5.44600000e+03],
       [2.85784480e-01, 5.46200000e+03],
       [2.86526870e-01, 5.16000000e+03],
       [2.86479433e-01, 5.59400000e+03],
       [2.86961877e-01, 4.90000000e+03],
       [2.87151204e-01, 4.99200000e+03],
       [2.87808993e-01, 5.10500000e+03],
       [2.87145621e-01, 4.61100000e+03],
       [2.86775708e-01, 3.79200000e+03],
       [2.86039622e-01, 3.96500000e+03],
       [2.85712331e-01, 4.01200000e+03],
       [2.85358328e-01, 4.01900000e+03],
       [2.85730709e-01, 4.40000000e+03],
       [2.84955951e-01, 5.50000000e+03],
       [2.85167772e-01, 3.79100000e+03],
       [2.85741355e-01, 3.24100000e+03],
       [2.85799824e-01, 3.03200000e+03],
       [2.85325727e-01, 2.96600000e+03],
       [2.85006076e-01, 2.98800000e+03],
       [2.853453

In [55]:
X_Tensor = np.expand_dims(X_combined, axis=0)

In [56]:
X_Tensor

array([[[2.84962344e-01, 8.82100000e+03],
        [2.87120556e-01, 9.28900000e+03],
        [2.87749032e-01, 1.05180000e+04],
        [2.87351760e-01, 1.00470000e+04],
        [2.86117538e-01, 6.45500000e+03],
        [2.85527424e-01, 5.44600000e+03],
        [2.85784480e-01, 5.46200000e+03],
        [2.86526870e-01, 5.16000000e+03],
        [2.86479433e-01, 5.59400000e+03],
        [2.86961877e-01, 4.90000000e+03],
        [2.87151204e-01, 4.99200000e+03],
        [2.87808993e-01, 5.10500000e+03],
        [2.87145621e-01, 4.61100000e+03],
        [2.86775708e-01, 3.79200000e+03],
        [2.86039622e-01, 3.96500000e+03],
        [2.85712331e-01, 4.01200000e+03],
        [2.85358328e-01, 4.01900000e+03],
        [2.85730709e-01, 4.40000000e+03],
        [2.84955951e-01, 5.50000000e+03],
        [2.85167772e-01, 3.79100000e+03],
        [2.85741355e-01, 3.24100000e+03],
        [2.85799824e-01, 3.03200000e+03],
        [2.85325727e-01, 2.96600000e+03],
        [2.85006076e-01, 2.9880000

In [ ]:
model = load_model("[...]")

In [58]:
predictions = model.predict(X_Tensor)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 984ms/step


In [59]:
predictions

array([[0.2409385]], dtype=float32)

In [60]:
decisive_sensibility = 0.5

predicted_classes = (predictions >= decisive_sensibility).astype(int)

In [61]:
print(predictions)
print(predicted_classes)

[[0.2409385]]
[[0]]
